In [3]:
!pip install torch torchvision pandas matplotlib
import torch
import torchvision
import torch.nn as nn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivanshcoding/1282-pokemon-139542-images-updated-pokedex-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\tanm6\.cache\kagglehub\datasets\shivanshcoding\1282-pokemon-139542-images-updated-pokedex-dataset\versions\10


In [1]:
import os
path = r"C:\Users\tanm6\.cache\kagglehub\datasets\shivanshcoding\1282-pokemon-139542-images-updated-pokedex-dataset\versions\10"
print(os.listdir(path))

['Pokedex Image Dataset', 'Pokemon Images for GAN', 'Pokemon Labelled Classification Images', 'updated_pokedex_dataset.csv']


In [6]:
import pandas as pd
pd.read_csv(path + '/updated_pokedex_dataset.csv')

,Num,Name,Type1,Type2,HP,Attack,Defense,SpAtk,SpDef,Speed,Generation,Legendary,Evolution
0,1,Abomasnow,Grass,Ice,90,92,75,92,85,60,4,False,Snover
1,2,Abra,Psychic,NaN,25,20,15,105,55,90,1,False,NaN
2,3,Absol,Dark,NaN,65,130,60,75,60,75,3,False,NaN
3,4,Accelgor,Bug,NaN,80,70,40,100,60,145,5,False,Shelmet
4,5,Aegislash,Steel,Ghost,60,50,140,50,140,60,6,False,Doublade
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1065,1066,Zoroark,Dark,NaN,60,105,60,120,60,105,5,False,Zorua
1066,1067,Zorua,Dark,NaN,40,65,40,80,40,65,5,False,NaN
1067,1068,Zubat,Poison,Flying,40,45,35,30,40,55,1,False,NaN
1068,1069,Zweilous,Dark,Dragon,72,85,70,65,70,58,5,False,Deino


In [6]:
from PIL import Image

imagePath = path + "/Pokemon Labelled Classification Images" + "/greninja" + "/greninja_0022.png"

image = Image.open(imagePath)
print(image.size)

(256, 256)


In [7]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor
from torch.utils.data import Subset

# tranform to a tensor
transform = Compose([
    Resize((256, 256)),
    ToTensor()
])

image_data = ImageFolder(path + "/Pokemon Labelled Classification Images", transform = transform)

valid_indices = []
for i, (filepath, _) in enumerate(image_data.samples):
    try:
        Image.open(filepath).verify()
        valid_indices.append(i)
    except Exception:
        pass

image_data = Subset(image_data, valid_indices)

# put the dataset into batches & shuffle (data wrapper)
image_loader = torch.utils.data.DataLoader(image_data, batch_size=32, shuffle=True)

print(f'Valid images: {len(image_data)}')

KeyboardInterrupt: 

In [8]:
# Creating the model

import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.efficientnet_b3(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1069)
model = model.to(device)  # move AFTER modifying

print(model.classifier)

Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1536, out_features=1069, bias=True)
)


In [12]:
# set up loss optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([
    {"params": model.features.parameters(), "lr": 1e-5},  # pretrained layers, train slowly
    {"params": model.classifier.parameters(), "lr": 1e-3} # new layer, train faster
])


In [19]:
epochs = 6

for e in range(epochs):
    # Loop through every image
    for (b, (X_train, y_train)) in enumerate(image_loader):
            
        X_train, y_train = X_train.to(device), y_train.to(device)

        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)

        # update parameters
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if(b % 5 == 0):
            print(f'Epoch: {e}, Batch: {b}, Loss: {loss.item()}')

    torch.save(model.state_dict(), f'model_epoch{e + 3}.pth')

Epoch: 0, Batch: 0, Loss: 0.17361709475517273
Epoch: 0, Batch: 5, Loss: 0.20958620309829712
Epoch: 0, Batch: 10, Loss: 0.4317420721054077
Epoch: 0, Batch: 15, Loss: 0.3652050197124481
Epoch: 0, Batch: 20, Loss: 0.43402019143104553
Epoch: 0, Batch: 25, Loss: 0.3429766297340393
Epoch: 0, Batch: 30, Loss: 0.5008311867713928
Epoch: 0, Batch: 35, Loss: 0.0615234449505806
Epoch: 0, Batch: 40, Loss: 0.7453858852386475
Epoch: 0, Batch: 45, Loss: 0.5353308916091919
Epoch: 0, Batch: 50, Loss: 0.26921936869621277
Epoch: 0, Batch: 55, Loss: 0.42361482977867126
Epoch: 0, Batch: 60, Loss: 0.42696312069892883
Epoch: 0, Batch: 65, Loss: 0.33829548954963684
Epoch: 0, Batch: 70, Loss: 0.1847875416278839
Epoch: 0, Batch: 75, Loss: 0.3357456922531128
Epoch: 0, Batch: 80, Loss: 0.3742557168006897
Epoch: 0, Batch: 85, Loss: 0.9160283207893372
Epoch: 0, Batch: 90, Loss: 0.6472377181053162
Epoch: 0, Batch: 95, Loss: 0.34875717759132385
Epoch: 0, Batch: 100, Loss: 0.3613707423210144
Epoch: 0, Batch: 105, Loss:

KeyboardInterrupt: 

In [18]:
torch.save(model.state_dict(), f'model_epoch2.pth')

In [10]:
# Testing
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.efficientnet_b3(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1069)
model.load_state_dict(torch.load('model_epoch6.pth', map_location=torch.device('cpu')))
model.eval()
model = model.to(device)


In [ ]:
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
import torch
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2

# Load class names from the dataset
dataset_path = r"C:\Users\tanm6\.cache\kagglehub\datasets\shivanshcoding\1282-pokemon-139542-images-updated-pokedex-dataset\versions\10"
temp_dataset = ImageFolder(dataset_path + "/Pokemon Labelled Classification Images")
class_names = temp_dataset.classes

transform = v2.Compose([
    v2.Resize((256, 256)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

# load and transform image
img = Image.open('IMG_6294.jpeg').convert('RGB')
img_tensor = transform(img).unsqueeze(0)  # add batch dimension
img_tensor = img_tensor.to(device)  # move tensor to same device as model

# predict
with torch.no_grad():
    output = model(img_tensor)
    predicted_idx = torch.argmax(output, dim=1).item()

import torch.nn.functional as F

probs = F.softmax(output, dim=1)
top5_probs, top5_indices = torch.topk(probs, 5)

print(f"Predicted Pokemon: {class_names[predicted_idx]}\n")
print("Top 5 predictions:")
for prob, idx in zip(top5_probs[0], top5_indices[0]):
    pokemon_name = class_names[idx.item()]
    print(f'{pokemon_name}: {prob.item()*100:.2f}%')


Predicted Pokemon: yamper

Top 5 predictions:
yamper: 21.36%
sandshrew: 16.74%
paras: 6.61%
emolga: 4.36%
kangaskhan: 3.99%
